# Homework 3 Work



In [7]:
# split ratings data into four partitions by timestamp.

# ratings.dat has timestamps in seconds since the epoch...
# need to divide based on these timestamps:

timestamp_ranges: list[list[str]] = [
    ["04/25/2000", "08/04/2000"],
    ["08/04/2000", "11/01/2000"],
    ["11/01/2000", "11/26/2000"],
    ["11/26/2000", "01/01/2027"]  # everything after 11/26/2000
]

# convert into seconds since the epoch (UTC midnight for each boundary)
from datetime import datetime, timezone

def to_epoch_seconds(date_str: str) -> int:
    dt = datetime.strptime(date_str, "%m/%d/%Y").replace(tzinfo=timezone.utc)
    return int(dt.timestamp())

timestamp_ranges_seconds: list[list[int]] = [
    [to_epoch_seconds(start), to_epoch_seconds(end)]
    for start, end in timestamp_ranges
]

timestamp_ranges_seconds

[[956620800, 965347200],
 [965347200, 973036800],
 [973036800, 975196800],
 [975196800, 1798761600]]

In [2]:
import pandas as pd

In [9]:
ratings = pd.read_csv("ml-1m/ratings.dat", sep="::", engine="python",
                    names=["user_id","movie_id","rating","timestamp"])

ratings.dtypes

user_id      int64
movie_id     int64
rating       int64
timestamp    int64
dtype: object

In [11]:
# split the ratings data by these timestamps

rating_date_partitions: dict[str, pd.DataFrame] = {}

for i, (start_ts, end_ts) in enumerate(timestamp_ranges_seconds):
    ratings_chunk = ratings[(ratings["timestamp"] >= start_ts) & (ratings["timestamp"] < end_ts)] 
    rating_date_partitions[f"ratings_{start_ts}-{end_ts}"] = (ratings_chunk)

for key, chunk in rating_date_partitions.items():
    chunk.info()
    print()
    path = f"ml-1m/{key}"
    chunk.to_csv(path)
    print(f"wrote csv to {path}")
    print()


<class 'pandas.core.frame.DataFrame'>
Index: 265435 entries, 692235 to 1000208
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   user_id    265435 non-null  int64
 1   movie_id   265435 non-null  int64
 2   rating     265435 non-null  int64
 3   timestamp  265435 non-null  int64
dtypes: int64(4)
memory usage: 10.1 MB

wrote csv to ml-1m/ratings_956620800-965347200

<class 'pandas.core.frame.DataFrame'>
Index: 235042 entries, 451428 to 994109
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   user_id    235042 non-null  int64
 1   movie_id   235042 non-null  int64
 2   rating     235042 non-null  int64
 3   timestamp  235042 non-null  int64
dtypes: int64(4)
memory usage: 9.0 MB

wrote csv to ml-1m/ratings_965347200-973036800

<class 'pandas.core.frame.DataFrame'>
Index: 245465 entries, 143479 to 993819
Data columns (total 4 columns):
 #   Column     Non-Null C

In [4]:
from pathlib import Path

Some utility functions adapted from HW2

```
You will run 4 iterations total. At each iteration:

Load the latest observation partition (the newly “arrived” data).
Handle newly added users (users that did not appear in earlier partitions).
Sample a random 30% subset of available users to compute user embeddings.
```

In [3]:
def read_ml1m(ml1m_folder: str="ml-1m") -> tuple[pd.DataFrame]:
    ratings = pd.read_csv(f"{ml1m_folder}/ratings.dat", sep="::", engine="python",
                      names=["user_id","movie_id","rating","timestamp"])
    movies  = pd.read_csv(f"{ml1m_folder}/movies.dat",  sep="::", engine="python",
                      names=["movie_id","title","genres"], encoding="latin-1")
    users   = pd.read_csv(f"{ml1m_folder}/users.dat", sep="::", engine="python",
                      names=["user_id", "gender", "age", "occupation", "zip"])
    return ratings,movies,users

ratings, movies, users = read_ml1m()

In [ ]:
def sample_users(users_df: pd.DataFrame, sample_pct: float) -> pd.DataFrame:
    """
    given a dataframe of users from the ml-1m dataset, returns a randomly sampled
    dataframe of `sample_pct`% of the users from `users_df`.
    """
    if not 0 < sample_pct <= 100:
        raise ValueError("sample_pct must a number >0 and <=100, got ", sample_pct)
    sample_frac = sample_pct / 100.0
    return users_df.sample(frac=sample_frac, random_state=42)

# I made this function so I can keep my sample consistent when generating embeddings, to
# be able to reuse my work for later tasks.
def sample_or_load_users(movie_lens_dir: str, sample_pct: float, users_df: pd.DataFrame = users) -> pd.DataFrame:
    """
    Attempts to load a sampled users file from ml-1m/users_sample_{sample_pct}.csv.
    If it does not exist, samples users_df and saves it to that path.
    """
    sample_file =  Path(movie_lens_dir) / f"users_sample_{int(sample_pct)}.csv"
    if sample_file.exists():
        print(f"{sample_file} already exists, loading df from csv")
        return pd.read_csv(sample_file)

    sampled_df = sample_users(users_df, sample_pct)
    sampled_df.to_csv(sample_file, index=False)
    return sampled_df

# global dataframe of all the users (user ratings) we have processed so far

# function to compute embeddings for a sample of users.

# Generative AI Disclosure

```
Please convert the timestamps strings into seconds since the epoch that are consistent with the schema for #file:ratings.dat defined in #file:README
```